In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
df = pd.read_csv("../data/raw/loan_data.csv")

In [8]:
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

In [9]:
X["loan_to_income"] = (
    X["loan_amnt"] / X["person_income"]
)

In [10]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [11]:
print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'loan_to_income']

Categorical features:
['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [14]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)

In [17]:
X_test_processed = preprocessor.transform(X_test)

In [18]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (36000, 14)
Processed training shape: (36000, 28)
Original testing shape: (9000, 14)
Processed testing shape: (9000, 28)


In [19]:
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['num__person_age', 'num__person_income', 'num__person_emp_exp',
       'num__loan_amnt', 'num__loan_int_rate', 'num__loan_percent_income',
       'num__cb_person_cred_hist_length', 'num__credit_score',
       'num__loan_to_income', 'cat__person_gender_female',
       'cat__person_gender_male', 'cat__person_education_Associate',
       'cat__person_education_Bachelor',
       'cat__person_education_Doctorate',
       'cat__person_education_High School',
       'cat__person_education_Master',
       'cat__person_home_ownership_MORTGAGE',
       'cat__person_home_ownership_OTHER',
       'cat__person_home_ownership_OWN',
       'cat__person_home_ownership_RENT',
       'cat__loan_intent_DEBTCONSOLIDATION', 'cat__loan_intent_EDUCATION',
       'cat__loan_intent_HOMEIMPROVEMENT', 'cat__loan_intent_MEDICAL',
       'cat__loan_intent_PERSONAL', 'cat__loan_intent_VENTURE',
       'cat__previous_loan_defaults_on_file_No',
       'cat__previous_loan_defaults_on_file_Yes'], dtype=object)

In [20]:
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=feature_names,
    index=X_train.index
)

In [21]:
X_train_processed_df.head()

,num__person_age,num__person_income,num__person_emp_exp,num__loan_amnt,num__loan_int_rate,num__loan_percent_income,num__cb_person_cred_hist_length,num__credit_score,num__loan_to_income,cat__person_gender_female,...,cat__person_home_ownership_OWN,cat__person_home_ownership_RENT,cat__loan_intent_DEBTCONSOLIDATION,cat__loan_intent_EDUCATION,cat__loan_intent_HOMEIMPROVEMENT,cat__loan_intent_MEDICAL,cat__loan_intent_PERSONAL,cat__loan_intent_VENTURE,cat__previous_loan_defaults_on_file_No,cat__previous_loan_defaults_on_file_Yes
6048,-0.622502,-0.256600,-0.561597,-0.823150,-1.687749,-0.800883,-0.480436,0.464300,-0.747584,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3346,-0.787921,-0.411269,-0.561597,0.224243,-0.001061,1.154322,-0.996078,0.026852,1.152138,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
17998,0.204591,1.902342,0.262920,0.065547,0.563408,-1.145920,0.808671,0.106388,-1.129378,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
24988,0.370009,0.186994,0.757630,-0.569236,0.832203,-0.915896,0.550850,0.981285,-0.890145,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
23231,0.204591,-0.089147,0.262920,-0.251845,-0.169058,-0.340835,0.550850,0.225692,-0.346550,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
